# Парсинг вакансий с trudvsem.ru

## Описание
Этот блокнот собирает IT-вакансии с портала «Работа в России» (trudvsem.ru) через открытый API.

## Что делает
1. Проверяет доступность API и структуру ответа.
2. Парсит вакансии по 86 ключевым словам.
3. Применяет строгий IT-фильтр (отсекает нерелевантные вакансии).
4. Сохраняет результат в `data/vacancies_trudvsem.csv`.

## Результат
~3 000 IT-вакансий с полным текстом описания (`duty` + `requirements`), навыками (`skills`), уровнем (`qualification`).

## Источник
API: https://opendata.trudvsem.ru/api/v1/vacancies

In [1]:
import re
import os
import time
import requests
import pandas as pd

os.makedirs("data", exist_ok=True)

## Шаг 1. Проверка структуры API

Убедимся, что API доступен и посмотрим, какие поля возвращает.

In [ ]:
import requests, json

url = "https://opendata.trudvsem.ru/api/v1/vacancies"
r = requests.get(url, params={"text": "программист", "limit": 1}, timeout=15)
print("Статус:", r.status_code)

if r.status_code == 200:
    data = r.json()
    vacs = data.get("results", {}).get("vacancies", [])
    if vacs:
        vac = vacs[0].get("vacancy", {})
        print("\nВсе поля первой вакансии:")
        for k, v in sorted(vac.items()):
            preview = str(v)[:100] if not isinstance(v, (dict, list)) else f"<{type(v).__name__}>"
            print(f"  {k}: {preview}")
    else:
        print("Пустой ответ")

## Шаг 2. Ключевые слова и строгий IT-фильтр

- `KEYWORDS` — 86 запросов к API (широкая сеть).
- `HARD_BLACKLIST` — явный мусор (учителя, врачи, монтажники).
- `STRICT_IT` — что считаем IT (программист, разработчик, data scientist, ...).
- `strict_is_it()` — итоговый фильтр: пропускает только IT-вакансии.

In [ ]:
KEYWORDS = [
    "программист", "разработчик", "developer", "python", "java", "golang",
    "javascript", "typescript", "c++", "c#", "php", "ruby", "rust", "kotlin", "swift",
    "1с", "битрикс", "bitrix", "frontend", "backend", "fullstack", "react", "vue", "angular",
    "django", "fastapi", "spring", "laravel", "node.js", ".net", "flutter",
    "системный администратор", "sysadmin", "sre", "linux", "docker", "kubernetes", "devops",
    "qa", "тестировщик", "автотест", "data engineer", "data analyst", "data scientist",
    "machine learning", "ml", "bi", "sql", "postgresql", "oracle", "etl", "big data",
    "системный аналитик", "бизнес-аналитик", "аналитик данных",
    "teamlead", "techlead", "scrum", "product manager", "project manager",
    "информационная безопасность", "кибербезопасность", "appsec", "devsecops", "pentest", "soc",
    "llm", "llmops", "rag", "genai", "computer vision", "nlp", "промпт", "mlops",
    "cloud architect", "data architect", "vector database",
    "архитектор программного обеспечения", "архитектор ит",
    "ux", "ui", "дизайнер интерфейсов", "верстальщик",
    "helpdesk", "техподдержка", "gamedev",
]

HARD_BLACKLIST = re.compile(
    r"электромонтер|электромонтажник|электромеханик"
    r"|инженер\s+по\s+метрологии|инженер\s+по\s+организации"
    r"|инженер\s+по\s+внедрению|инженер\s+по\s+промышленной"
    r"|химик-аналитик|маркетолог-аналитик|финансовый\s+аналитик"
    r"|экономист|бухгалтер|юрист|юрисконсульт|кадр|делопроизвод"
    r"|оперуполномоченн|полицейск|росгвард|военнослуж"
    r"|антифрод|анти-фрод"
    r"|энергетик|гидрогеолог|геодезист|геолог|маркшейдер"
    r"|прораб|сметчик|строительн"
    r"|медицинск|врач|медсестр|фельдшер|санитар"
    r"|воспитател|учител|преподавател|педагог"
    r"|водител|машинист|тракторист"
    r"|повар|официант|курьер|охранник|уборщик|дворник"
    r"|продавец|кассир|мерчендайзер|грузчик|кладовщик|комплектовщик",
    re.IGNORECASE,
)

STRICT_IT = re.compile(
    r"программист|разработчик|developer|python|java|golang|c\+\+|c#|javascript|typescript"
    r"|1с|1c|битрикс|bitrix|ios|android|веб-разработ|web-разработ|frontend|backend|fullstack|full-stack"
    r"|data\s*scien|data\s*engineer|data\s*analyst|ml\s*engineer|mlops|nlp|computer\s*vision"
    r"|\bllm\b|нейросет|нейрон|\bml\b|\bai\b"
    r"|аналитик\s+данных|аналитик\s+bi|bi-аналитик|продуктовый\s+аналитик"
    r"|devops|sre|системный\s+администратор|sysadmin|сетевой\s+администратор"
    r"|dba|администратор\s+баз\s+данных|database"
    r"|qa|тестировщик|test\s+engineer|автотест"
    r"|информационной\s+безопасности|кибербезопасн|appsec|devsecops|pentest|пентест"
    r"|soc-аналитик|soc\s+аналитик"
    r"|архитектор\s+(по\s+)?(ит|it|систем|программ|данных|решений)"
    r"|product\s+manager|продакт|проектный\s+менеджер|it-проект"
    r"|scrum|agile|product\s+owner"
    r"|ux|ui|дизайнер\s+интерфейс|верстальщик|frontend-разработ"
    r"|\bit-специалист|\bit\s+специалист|инженер-программист|техник-программист"
    r"|специалист\s+по\s+защите\s+информации|инженер\s+по\s+защите\s+информации"
    r"|системный\s+инженер|инженер\s+по\s+информационн"
    r"|баз\s+данных|бд\s+администратор",
    re.IGNORECASE,
)

def strict_is_it(title):
    """Проверяет, является ли вакансия IT по названию."""
    t = str(title).strip()
    if not t:
        return False
    if HARD_BLACKLIST.search(t):
        return False
    return bool(STRICT_IT.search(t))

# Быстрая проверка
print("strict_is_it('Программист 1С') =", strict_is_it("Программист 1С"))
print("strict_is_it('Учитель информатики') =", strict_is_it("Учитель информатики"))
print("strict_is_it('Электромонтер') =", strict_is_it("Электромонтер"))

## Шаг 3. Парсер

Функция `fetch_trudvsem_vacancies`:
- Проходит по 86 ключевым словам.
- Для каждого — пагинация (до 100 страниц по 100 вакансий).
- Применяет `strict_is_it` к названию.
- Сохраняет полный набор полей: `duty`, `requirements`, `qualification`, `skills`, `salary`, `company`, `date_create`, ...

In [ ]:
def fetch_trudvsem_vacancies(existing_ids=None):
    if existing_ids is None:
        existing_ids = set()
    base_url = "https://opendata.trudvsem.ru/api/v1/vacancies"
    vacancies_dict = {}

    for i, kw in enumerate(KEYWORDS, 1):
        page = 0
        added_for_kw = 0
        while page < 100:
            params = {"text": kw, "offset": page, "limit": 100}
            try:
                res = requests.get(base_url, params=params, timeout=15)
                if res.status_code != 200:
                    break
                results = res.json().get("results", {}).get("vacancies", [])
                if not results:
                    break

                for item in results:
                    vac = item.get("vacancy", {})
                    v_id = f"trud_{vac.get('id')}"
                    if not v_id or v_id in existing_ids or v_id in vacancies_dict:
                        continue

                    title = vac.get("job-name", "")
                    if not strict_is_it(title):
                        continue

                    sal_from = vac.get("salary_min", 0) or 0
                    sal_to = vac.get("salary_max", 0) or 0
                    salary = (sal_from + sal_to) / 2 if (sal_from and sal_to) else (sal_from or sal_to)
                    if not salary or salary <= 0 or salary > 1_000_000:
                        continue

                    duty = vac.get("duty", "") or ""
                    requirements = vac.get("requirements", "") or ""
                    qualification_text = vac.get("qualification", "") or ""
                    skills = vac.get("skills", []) or []

                    vacancies_dict[v_id] = {
                        "id": v_id,
                        "title": title,
                        "duty": duty,
                        "requirements": requirements,
                        "qualification": qualification_text,
                        "skills": "|".join(skills) if isinstance(skills, list) else str(skills),
                        "description": f"{duty} {requirements}".strip(),
                        "salary": salary,
                        "salary_min": sal_from,
                        "salary_max": sal_to,
                        "salary_text": vac.get("salary", ""),
                        "currency": vac.get("currency", ""),
                        "date_create": vac.get("creation-date", None),
                        "date_modify": vac.get("date_modify", None),
                        "region_name": vac.get("region", {}).get("name", ""),
                        "company_name": vac.get("company", {}).get("name", ""),
                        "company_inn": vac.get("company", {}).get("inn", ""),
                        "schedule": vac.get("schedule", ""),
                        "work_places": vac.get("work_places", None),
                        "typical_position": vac.get("typicalPosition", ""),
                        "code_profession": vac.get("code_profession", None),
                        "source": "trudvsem.ru",
                    }
                    added_for_kw += 1

                page += 1
                time.sleep(0.02)
            except Exception as e:
                print(f"  Ошибка на '{kw}', page {page}: {e}")
                break

        print(f"[{i}/{len(KEYWORDS)}] '{kw}': +{added_for_kw}, всего: {len(vacancies_dict)}")

    return vacancies_dict

## Шаг 4. Запуск парсинга

⚠️ Занимает **не меньше 30 минут** (86 ключевых слов, пагинация). Можно прервать и запустить снова — функция поддерживает `existing_ids` (но в этом блокноте мы парсим с нуля).

In [ ]:
dict_trud = fetch_trudvsem_vacancies()
df = pd.DataFrame(list(dict_trud.values()))

print(f"\nВсего уникальных IT-вакансий: {len(df)}")
print(f"С датами: {df['date_create'].notna().sum()}")
print(f"С qualification: {(df['qualification'] != '').sum()}")
print(f"С skills: {(df['skills'] != '').sum()}")
print(f"С company_name: {df['company_name'].notna().sum()}")
print(f"\nМедиана зарплаты: {df['salary'].median():.0f}")

In [ ]:
df.to_csv("data/vacancies_trudvsem.csv", index=False, encoding="utf-8-sig")
print(f"✓ Сохранено: data/vacancies_trudvsem.csv ({len(df)} строк)")

## Результат

- **Файл:** `data/vacancies_trudvsem.csv`
- **Строк:** ~3 000
- **Колонок:** 22
- **Поля:** `id`, `title`, `duty`, `requirements`, `qualification`, `skills`, `description`, `salary`, `salary_min`, `salary_max`, `region_name`, `company_name`, `date_create`, ...

## Следующий шаг
Перейти к `02_loading_kaggle.ipynb` — загрузка датасетов hh.ru.